## Imports

In [1]:
import pickle
from pathlib import Path

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import seaborn

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from scipy.stats import pearsonr
from statsmodels.stats.multitest import multipletests

import PyWGCNA

/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/daniel.hosomi/csvd-projects/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configurações

In [2]:
PROCESSED_DIR = Path("../../data/interim")
DESEQ_DIR = Path("../../data/interim/deseq2")
LOGCPM_PATH = PROCESSED_DIR / "microplastic_logcpm_filtered.csv"
METADATA_PATH = PROCESSED_DIR / "microplastic_metadata.csv"
# Input: log2(normed_counts + 1) gerado pelo notebook 002
WGCNA_INPUT_PATH = PROCESSED_DIR / "microplastic_log2norm_wgcna_input.csv"

# Saídas
WGCNA_DIR = Path("../../data/interim/wgcna")
WGCNA_DIR.mkdir(parents=True, exist_ok=True)

GENE_MODULES_PATH              = WGCNA_DIR / "wgcna_gene_modules.csv"
MODULE_EIGENGENES_PATH         = WGCNA_DIR / "wgcna_module_eigengenes.csv"
MODULE_EIGENGENES_TREATED_PATH = WGCNA_DIR / "wgcna_module_eigengenes_treated.csv"
MODULE_TRAIT_CORR_PATH         = WGCNA_DIR / "wgcna_module_trait_correlations_treated.csv"
MODULE_TRAIT_PVAL_PATH         = WGCNA_DIR / "wgcna_module_trait_pvalues_treated.csv"
MODULE_TRAIT_PADJ_PATH         = WGCNA_DIR / "wgcna_module_trait_padj_treated.csv"
MODULE_SUMMARY_PATH            = WGCNA_DIR / "wgcna_module_summary.csv"
HEATMAP_PATH                   = WGCNA_DIR / "wgcna_module_trait_heatmap_treated.png"

# Parâmetros-chave do WGCNA — altere aqui para reexecutar com diferentes valores.
# O nome do modelo e o path do cache são derivados automaticamente dos parâmetros,
# evitando sobrescrever resultados anteriores.
MIN_MODULE_SIZE = 20
ME_DISS_THRES   = 0.25

WGCNA_NAME       = f"microplastic_wgcna_minmod{MIN_MODULE_SIZE}_meds{int(ME_DISS_THRES * 100)}"
WGCNA_CACHE_PATH = WGCNA_DIR / f"{WGCNA_NAME}.p"

## Carregamento dos Dados de DEG

In [3]:
log2norm_df = pd.read_csv(WGCNA_INPUT_PATH)
metadata_df = pd.read_csv(METADATA_PATH)

print("log2norm:", log2norm_df.shape)
print("metadata:", metadata_df.shape)

display(log2norm_df.head())
display(metadata_df.head())

log2norm: (12174, 25)
metadata: (24, 9)


,gene_id,CTR_1,CTR_2,CTR_3,MA1_1,MA1_2,MA1_3,MB1_1,MB1_2,MB1_3,...,MD1_3,MA100_1,MA100_2,MA100_3,MB100_1,MB100_2,MB100_3,MC100_1,MC100_2,MC100_3
0,ENSG00000000003,8.495462,8.450681,8.523000,8.361693,7.986612,8.218316,8.495056,8.136695,8.608448,...,8.256791,8.322836,8.369257,8.097211,8.272033,8.003671,8.616483,8.234527,8.321764,8.461571
1,ENSG00000000419,9.602250,8.919754,9.676562,9.491573,9.459700,9.228012,9.289487,9.577056,9.501974,...,9.781742,9.421966,9.663061,9.276540,9.361850,9.593103,9.537103,9.461464,9.762872,9.298874
2,ENSG00000000457,4.818618,6.950606,5.152469,4.995176,5.243716,4.919546,5.047678,5.071817,5.001960,...,5.447777,4.882649,5.203948,2.789318,4.416598,5.488284,5.716236,5.347044,5.944513,5.911113
3,ENSG00000000460,5.259300,4.897526,3.138399,2.524263,2.720120,4.958863,4.359242,5.551225,5.479900,...,0.000000,4.835425,2.160350,0.000000,4.064362,3.442941,4.540396,3.657667,4.300867,4.913910
4,ENSG00000001036,9.949776,9.975952,10.080303,9.906705,9.953352,10.599921,10.073621,10.080813,9.783203,...,10.036049,10.022790,9.818998,9.955213,9.849547,9.953890,10.260942,10.283438,10.236575,10.253528


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,sample_id,group,replicate
0,control,NaN,NaN,0.0,True,control,CTR_1,CTR,1
1,control,NaN,NaN,0.0,True,control,CTR_2,CTR,2
2,control,NaN,NaN,0.0,True,control,CTR_3,CTR,3
3,polystyrene,1.0,1000.0,0.1,False,treated,MA1_1,MA1,1
4,polystyrene,1.0,1000.0,0.1,False,treated,MA1_2,MA1,2


## Construção do WGCNA

In [4]:
# Preparação da matriz de expressão para PyWGCNA.
# A matriz em arquivo está em genes x amostras.
# Para o construtor WGCNA do PyWGCNA, deve-se usar amostras x genes.

sample_ids = metadata_df["sample_id"].tolist()
expr_sample_cols = [c for c in log2norm_df.columns if c != "gene_id"]

missing_in_expr = sorted(set(sample_ids) - set(expr_sample_cols))
missing_in_meta = sorted(set(expr_sample_cols) - set(sample_ids))

if missing_in_expr or missing_in_meta:
    raise ValueError(
        f"Inconsistência entre expressão e metadata.\n"
        f"Ausentes na expressão: {missing_in_expr}\n"
        f"Ausentes no metadata: {missing_in_meta}"
    )

# Reordena as colunas conforme a ordem do metadata
log2norm_df = log2norm_df[["gene_id"] + sample_ids]

# Formato esperado para WGCNA: amostras x genes
expr_wgcna = log2norm_df.set_index("gene_id").T
expr_wgcna.index.name = "sample_id"

# Metadata alinhado
metadata = metadata_df.set_index("sample_id").loc[expr_wgcna.index].copy()

print("Matriz para WGCNA:", expr_wgcna.shape)
print("Metadata alinhado:", metadata.shape)

display(expr_wgcna.iloc[:5, :5])
display(metadata.head())

Matriz para WGCNA: (24, 12174)
Metadata alinhado: (24, 8)


gene_id,ENSG00000000003,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000001036
sample_id,,,,,
CTR_1,8.495462,9.602250,4.818618,5.259300,9.949776
CTR_2,8.450681,8.919754,6.950606,4.897526,9.975952
CTR_3,8.523000,9.676562,5.152469,3.138399,10.080303
MA1_1,8.361693,9.491573,4.995176,2.524263,9.906705
MA1_2,7.986612,9.459700,5.243716,2.720120,9.953352


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate
sample_id,,,,,,,,
CTR_1,control,NaN,NaN,0.0,True,control,CTR,1
CTR_2,control,NaN,NaN,0.0,True,control,CTR,2
CTR_3,control,NaN,NaN,0.0,True,control,CTR,3
MA1_1,polystyrene,1.0,1000.0,0.1,False,treated,MA1,1
MA1_2,polystyrene,1.0,1000.0,0.1,False,treated,MA1,2


In [5]:
# Preparação do metadata para PyWGCNA.

# Corrige possíveis problemas de tipo ao ler CSV
if metadata["is_control"].dtype == object:
    metadata["is_control"] = metadata["is_control"].astype(str).str.lower().map({
        "true": True,
        "false": False
    })

metadata["particle_size_um"] = pd.to_numeric(metadata["particle_size_um"], errors="coerce")
metadata["particle_size_nm"] = pd.to_numeric(metadata["particle_size_nm"], errors="coerce")
metadata["concentration_gL"] = pd.to_numeric(metadata["concentration_gL"], errors="coerce")

display(metadata.dtypes)
display(metadata.head())

particle_type        object
particle_size_um    float64
particle_size_nm    float64
concentration_gL    float64
is_control             bool
treatment_status     object
group                object
replicate             int64
dtype: object

,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate
sample_id,,,,,,,,
CTR_1,control,NaN,NaN,0.0,True,control,CTR,1
CTR_2,control,NaN,NaN,0.0,True,control,CTR,2
CTR_3,control,NaN,NaN,0.0,True,control,CTR,3
MA1_1,polystyrene,1.0,1000.0,0.1,False,treated,MA1,1
MA1_2,polystyrene,1.0,1000.0,0.1,False,treated,MA1,2


In [6]:
# Construção do objeto WGCNA

pyw = PyWGCNA.WGCNA(
    name=WGCNA_NAME,                    # derivado dos parâmetros — veja Cell 3
    species="human",                     # usado para anotação funcional (GO, KEGG)
    geneExp=expr_wgcna,                  # matriz de expressão: amostras × genes (log2norm)
    sampleInfo=metadata,                 # metadata indexado por sample_id
    outputPath=str(WGCNA_DIR) + "/",    # trailing slash obrigatório — PyWGCNA concatena strings
    save=True,                           # salva resultados intermediários automaticamente

    # Filtro de expressão mínima — já filtramos no notebook 001, então desativado aqui
    TPMcutoff=0,

    # Poderes a testar para escolha do soft-threshold β (escala 1–10 contínuo, 12–20 de 2 em 2)
    # O WGCNA seleciona o menor poder que atinge RsquaredCut na topologia livre de escala
    powers=list(range(1, 11)) + list(range(12, 22, 2)),
    RsquaredCut=0.8,                     # R² mínimo do ajuste de topologia livre de escala
    MeanCut=100,                         # conectividade média máxima (evita redes super-conectadas)

    # "signed hybrid": preserva o sinal da correlação (positivo ≠ negativo), mas trata
    # correlações negativas de forma mais suave que "signed" puro — padrão recomendado
    # para dados de RNA-seq onde co-regulação negativa é biologicamente relevante
    networkType="signed hybrid",
    TOMType="signed",                    # TOM com sinal: considera direção da correlação no overlap

    # Módulo com menos de MIN_MODULE_SIZE genes é dissolvido e seus genes vão para dimgrey.
    # Reduzir esse valor permite que clusters menores formem módulos próprios.
    minModuleSize=MIN_MODULE_SIZE,

    # Módulos com correlação entre eigengenes > 1 - ME_DISS_THRES são fundidos.
    # Valor ligeiramente maior (0.25) = fusão menos agressiva → mais módulos distintos preservados.
    MEDissThres=ME_DISS_THRES,
)

Saving data to be True, checking requirements ...


In [7]:
if WGCNA_CACHE_PATH.exists():
    print(f"Carregando WGCNA do cache: {WGCNA_CACHE_PATH}")
    # pyw = PyWGCNA.readWGCNA("microplastic_wgcna", outputPath=str(WGCNA_DIR))
    with open(WGCNA_CACHE_PATH, "rb") as f:
        pyw = pickle.load(f)
    print("WGCNA carregado com sucesso.")
else:
    print("Cache não encontrado — executando WGCNA (pode levar vários minutos)...")
    pyw.runWGCNA()
    if hasattr(pyw, "saveWGCNA"):
        pyw.saveWGCNA()
    print(f"WGCNA executada e cache salvo em: {WGCNA_CACHE_PATH}")

Carregando WGCNA do cache: ../../data/interim/wgcna/microplastic_wgcna_minmod20_meds25.p
WGCNA carregado com sucesso.


In [8]:
# Diagnóstico do soft-thresholding power selecionado pelo PyWGCNA
print(f"Soft-thresholding power selecionado: {pyw.power}")
print(f"(R² target: {pyw.RsquaredCut})")

Soft-thresholding power selecionado: 4
(R² target: 0.8)


## Inspeção dos Resultados

In [9]:
# Verificar os principais atributos do objeto PyWGCNA após a execução

print("type(pyw.datExpr):", type(getattr(pyw, "datExpr", None)))
print("shape datExpr:", getattr(getattr(pyw, "datExpr", None), "shape", None))

if hasattr(pyw, "datExpr") and pyw.datExpr is not None:
    print("var columns:", pyw.datExpr.var.columns)
    display(pyw.datExpr.var.head())

type(pyw.datExpr): <class 'anndata._core.anndata.AnnData'>
shape datExpr: (24, 12174)
var columns: Index(['dynamicColors', 'moduleColors', 'moduleLabels'], dtype='object')


,dynamicColors,moduleColors,moduleLabels
gene_id,,,
ENSG00000000003,dimgrey,dimgrey,11
ENSG00000000419,silver,silver,29
ENSG00000000457,dimgrey,dimgrey,11
ENSG00000000460,dimgrey,dimgrey,11
ENSG00000001036,lightcoral,lightcoral,14


In [10]:
# Extração de genes e módulos finais obtidos com WGCNA

if not hasattr(pyw, "datExpr") or pyw.datExpr is None:
    raise ValueError("pyw.datExpr não está disponível.")

gene_var = pyw.datExpr.var.copy()

expected_cols = {"dynamicColors", "moduleColors", "moduleLabels"}
missing_cols = expected_cols - set(gene_var.columns)

if missing_cols:
    raise ValueError(f"Colunas esperadas ausentes em pyw.datExpr.var: {missing_cols}")

gene_modules_df = (
    gene_var
    .reset_index()
    .rename(columns={"index": "gene_id"})
    [["gene_id", "dynamicColors", "moduleColors", "moduleLabels"]]
    .copy()
)

gene_modules_df = gene_modules_df.rename(columns={
    "dynamicColors": "dynamic_module",
    "moduleColors": "module",
    "moduleLabels": "module_label"
})

gene_modules_df.to_csv(GENE_MODULES_PATH, index=False)

print("Tabela gene -> módulo salva em:", GENE_MODULES_PATH)
print("Dimensões:", gene_modules_df.shape)

display(gene_modules_df.head())
display(
    gene_modules_df["module"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "module", "module": "n_genes"})
)

# NOTA: O módulo 'dimgrey' representa genes não atribuídos a nenhum módulo de
# coexpressão (equivalente ao módulo "grey" do WGCNA em R). Não há threshold
# oficialmente estabelecido na literatura para a proporção aceitável de genes
# cinza (Bioconductor WGCNA FAQ). Um valor alto indica que muitos genes ficaram
# fora dos módulos interpretativos — o ajuste de minModuleSize, MEDissThres ou
# soft-threshold power pode aumentar a cobertura biológica.
n_dimgrey = (gene_modules_df["module"] == "dimgrey").sum()
n_total = len(gene_modules_df)
pct_unassigned = 100 * n_dimgrey / n_total
print(f"\n⚠️ Módulo 'dimgrey' (não atribuídos): {n_dimgrey} genes ({pct_unassigned:.1f}% do total)")
print("Não há threshold oficial — um valor alto reduz a cobertura de enriquecimento funcional.")

Tabela gene -> módulo salva em: ../../data/interim/wgcna/wgcna_gene_modules.csv
Dimensões: (12174, 4)


,gene_id,dynamic_module,module,module_label
0,ENSG00000000003,dimgrey,dimgrey,11
1,ENSG00000000419,silver,silver,29
2,ENSG00000000457,dimgrey,dimgrey,11
3,ENSG00000000460,dimgrey,dimgrey,11
4,ENSG00000001036,lightcoral,lightcoral,14


,n_genes,count
0,dimgrey,3847
1,darkgrey,3279
2,silver,2145
3,gainsboro,1116
4,maroon,340
5,white,316
6,lightcoral,192
7,firebrick,171
8,brown,143
9,red,116



⚠️ Módulo 'dimgrey' (não atribuídos): 3847 genes (31.6% do total)
Não há threshold oficial — um valor alto reduz a cobertura de enriquecimento funcional.


In [11]:
# Cálculo dos module eigengenes a partir dos módulos finais.
#
# NOTA: Calculamos eigengenes manualmente via PCA ao invés de usar pyw.getEigengenes()
# porque a API do PyWGCNA retorna eigengenes na orientação interna do objeto AnnData,
# dificultando o alinhamento com nosso metadata indexado por sample_id.
# O resultado é matematicamente equivalente: o 1º componente principal do bloco de
# expressão de cada módulo, com sinal corrigido para correlacionar positivamente com
# o perfil médio do módulo.

module_eigengenes = {}

for module_name, module_df in gene_modules_df.groupby("module"):
    genes = [g for g in module_df["gene_id"].tolist() if g in expr_wgcna.columns]

    if len(genes) == 0:
        continue

    X = expr_wgcna[genes].copy()

    # Se o módulo tiver apenas 1 gene, usa o próprio perfil
    if X.shape[1] == 1:
        eigengene = X.iloc[:, 0].values.astype(float)
    else:
        pca = PCA(n_components=1, random_state=42)
        eigengene = pca.fit_transform(X.values).ravel()

        # Ajusta o sinal para ficar coerente com o perfil médio do módulo
        mean_profile = X.mean(axis=1).values
        corr_sign = np.corrcoef(eigengene, mean_profile)[0, 1]
        if pd.notna(corr_sign) and corr_sign < 0:
            eigengene = -eigengene

    module_eigengenes[f"ME_{module_name}"] = eigengene

module_eigengenes_df = pd.DataFrame(
    module_eigengenes,
    index=expr_wgcna.index
)

module_eigengenes_df.index.name = "sample_id"
module_eigengenes_df.to_csv(MODULE_EIGENGENES_PATH)

print("Module eigengenes salvos em:", MODULE_EIGENGENES_PATH)
print("Dimensões:", module_eigengenes_df.shape)

display(module_eigengenes_df.head())

Module eigengenes salvos em: ../../data/interim/wgcna/wgcna_module_eigengenes.csv
Dimensões: (24, 32)


,ME_antiquewhite,ME_bisque,ME_blanchedalmond,ME_brown,ME_burlywood,ME_chocolate,ME_coral,ME_darkgrey,ME_darkorange,ME_darkred,...,ME_peachpuff,ME_peru,ME_red,ME_saddlebrown,ME_sandybrown,ME_seashell,ME_sienna,ME_silver,ME_tan,ME_white
sample_id,,,,,,,,,,,,,,,,,,,,,
CTR_1,1.221235,1.184029,-0.505317,1.220678,0.999288,1.916591,0.728734,-10.318899,-1.046436,1.137659,...,0.310090,-0.013152,-0.140782,-0.745005,1.114804,1.854321,0.542004,13.925312,0.069283,-0.400751
CTR_2,3.757427,1.262612,-1.469915,0.974999,-0.636762,3.612219,-3.250010,-16.065127,1.761363,1.866965,...,-1.411649,4.281665,3.106565,1.850818,0.622786,-0.552037,2.587997,3.028996,-3.757274,-11.798660
CTR_3,-2.679445,-6.395802,1.077578,0.543351,-3.563805,-7.406817,-1.499689,-18.430496,1.494609,0.721162,...,0.779345,-1.214200,-0.642754,0.891494,1.815529,1.030539,1.259534,13.121090,0.303769,-7.423641
MA1_1,-0.877890,2.327073,1.760794,0.591744,2.543267,-2.419743,-0.231744,0.497531,3.115813,0.430484,...,4.002721,2.928066,1.399091,0.960467,0.545700,4.697996,2.191988,1.661865,1.639240,1.280734
MA1_2,-2.675375,1.365602,0.717863,1.465767,2.017885,0.568306,-2.322539,2.579179,-1.538518,-0.199851,...,-7.810588,-6.095901,1.541981,2.015989,2.212055,-4.703557,1.549874,-2.779813,1.387043,-0.290838


In [12]:
# Filtrar apenas as amostras tratadas para correlação módulo-traço

metadata_treated = metadata[metadata["is_control"] == False].copy()
module_eigengenes_treated = module_eigengenes_df.loc[metadata_treated.index].copy()

print("Amostras totais:", metadata.shape[0])
print("Amostras tratadas:", metadata_treated.shape[0])
print("Module eigengenes (tratadas):", module_eigengenes_treated.shape)

display(metadata_treated.head())
display(module_eigengenes_treated.head())

Amostras totais: 24
Amostras tratadas: 21
Module eigengenes (tratadas): (21, 32)


,particle_type,particle_size_um,particle_size_nm,concentration_gL,is_control,treatment_status,group,replicate
sample_id,,,,,,,,
MA1_1,polystyrene,1.0,1000.0,0.10,False,treated,MA1,1
MA1_2,polystyrene,1.0,1000.0,0.10,False,treated,MA1,2
MA1_3,polystyrene,1.0,1000.0,0.10,False,treated,MA1,3
MB1_1,polystyrene,1.0,1000.0,0.01,False,treated,MB1,1
MB1_2,polystyrene,1.0,1000.0,0.01,False,treated,MB1,2


,ME_antiquewhite,ME_bisque,ME_blanchedalmond,ME_brown,ME_burlywood,ME_chocolate,ME_coral,ME_darkgrey,ME_darkorange,ME_darkred,...,ME_peachpuff,ME_peru,ME_red,ME_saddlebrown,ME_sandybrown,ME_seashell,ME_sienna,ME_silver,ME_tan,ME_white
sample_id,,,,,,,,,,,,,,,,,,,,,
MA1_1,-0.877890,2.327073,1.760794,0.591744,2.543267,-2.419743,-0.231744,0.497531,3.115813,0.430484,...,4.002721,2.928066,1.399091,0.960467,0.545700,4.697996,2.191988,1.661865,1.639240,1.280734
MA1_2,-2.675375,1.365602,0.717863,1.465767,2.017885,0.568306,-2.322539,2.579179,-1.538518,-0.199851,...,-7.810588,-6.095901,1.541981,2.015989,2.212055,-4.703557,1.549874,-2.779813,1.387043,-0.290838
MA1_3,-0.562821,0.284389,1.921503,3.402404,-1.106855,-0.290000,1.224502,-1.174953,0.344007,1.125130,...,2.005658,0.964602,-0.130911,0.536240,0.709391,1.921164,1.247294,15.905508,1.167152,7.919318
MB1_1,1.569824,-2.673341,0.907818,2.205937,2.234922,0.740718,-0.070380,15.677543,-1.722652,-5.157010,...,0.109657,2.408711,-2.807665,-0.976410,2.091487,0.197120,1.818188,-1.343452,1.606765,1.839650
MB1_2,-2.606125,-1.911984,1.921859,-1.599608,2.008707,0.475328,-2.187875,2.207072,1.231139,-5.037548,...,2.684256,0.996932,-4.233909,0.625548,2.488562,-5.125086,0.599029,-10.274094,1.178715,0.628513


In [13]:
# Montar traços de interesse para correlação

metadata_treated = metadata_treated.copy()

# Variável binária principal para distinguir 100 nm de 1 µm
metadata_treated["is_100nm"] = (metadata_treated["particle_size_nm"] == 100).astype(int)

# Dummies por grupo para leitura mais fina no heatmap
group_dummies_treated = pd.get_dummies(metadata_treated["group"], prefix="group")

# Traits não redundantes
traits_numeric_treated = pd.concat([
    metadata_treated[[
        "particle_size_um",
        "concentration_gL",
        "is_100nm",
    ]],
    group_dummies_treated
], axis=1)

print("Traits numéricos (tratadas):", traits_numeric_treated.shape)
display(traits_numeric_treated.head())

Traits numéricos (tratadas): (21, 10)


,particle_size_um,concentration_gL,is_100nm,group_MA1,group_MA100,group_MB1,group_MB100,group_MC1,group_MC100,group_MD1
sample_id,,,,,,,,,,
MA1_1,1.0,0.10,0,True,False,False,False,False,False,False
MA1_2,1.0,0.10,0,True,False,False,False,False,False,False
MA1_3,1.0,0.10,0,True,False,False,False,False,False,False
MB1_1,1.0,0.01,0,False,False,True,False,False,False,False
MB1_2,1.0,0.01,0,False,False,True,False,False,False,False


In [14]:
# Correlação módulo-traço

def safe_pearsonr(x, y):
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)

    mask = x.notna() & y.notna()

    if mask.sum() < 3:
        return np.nan, np.nan

    if x[mask].nunique() < 2 or y[mask].nunique() < 2:
        return np.nan, np.nan

    r, p = pearsonr(x[mask], y[mask])
    return r, p

corr_matrix_treated = pd.DataFrame(
    index=module_eigengenes_treated.columns,
    columns=traits_numeric_treated.columns,
    dtype=float
)

pval_matrix_treated = pd.DataFrame(
    index=module_eigengenes_treated.columns,
    columns=traits_numeric_treated.columns,
    dtype=float
)

for me_col in module_eigengenes_treated.columns:
    for trait_col in traits_numeric_treated.columns:
        r, p = safe_pearsonr(
            module_eigengenes_treated[me_col],
            traits_numeric_treated[trait_col]
        )
        corr_matrix_treated.loc[me_col, trait_col] = r
        pval_matrix_treated.loc[me_col, trait_col] = p

print("Matriz de correlação:", corr_matrix_treated.shape)
print("Matriz de p-values:", pval_matrix_treated.shape)

display(corr_matrix_treated)
display(pval_matrix_treated)

Matriz de correlação: (32, 10)
Matriz de p-values: (32, 10)


,particle_size_um,concentration_gL,is_100nm,group_MA1,group_MA100,group_MB1,group_MB100,group_MC1,group_MC100,group_MD1
ME_antiquewhite,-0.110407,-0.045724,0.110407,-0.267628,0.099140,-0.237109,-0.138579,0.050200,0.195578,0.298398
ME_bisque,0.059090,0.041020,-0.059090,0.206388,-0.231067,-0.089884,0.004578,-0.198067,0.142924,0.165128
ME_blanchedalmond,0.118667,0.530410,-0.118667,0.271246,0.296859,-0.110171,-0.322800,-0.212841,-0.141880,0.219587
ME_brown,0.320837,-0.264797,-0.320837,0.118707,-0.511990,-0.005717,-0.025973,0.183270,0.084231,0.157472
ME_burlywood,-0.127795,0.028652,0.127795,0.186918,-0.158725,0.186687,0.294648,-0.478533,0.044807,-0.075801
ME_chocolate,-0.002155,-0.341832,0.002155,-0.195789,-0.269411,0.288339,0.139667,-0.126891,0.132792,0.031293
ME_coral,0.229163,-0.317872,-0.229163,-0.072619,-0.492753,-0.181631,0.088842,0.190823,0.079825,0.387513
ME_darkgrey,0.630282,0.216641,-0.630282,-0.050810,0.146106,0.282469,-0.319865,0.319346,-0.717594,0.340348
ME_darkorange,-0.060787,0.048028,0.060787,0.143988,-0.106199,-0.211396,0.203709,-0.060236,-0.011544,0.041678
ME_darkred,-0.563291,0.280868,0.563291,0.106192,0.509717,-0.670177,-0.050428,0.197438,0.337325,-0.430067


,particle_size_um,concentration_gL,is_100nm,group_MA1,group_MA100,group_MB1,group_MB100,group_MC1,group_MC100,group_MD1
ME_antiquewhite,0.633769,0.843979,0.633769,0.240845,0.668976,0.300721,0.549129,0.828913,0.395536,0.188889
ME_bisque,0.799168,0.859869,0.799168,0.369401,0.313567,0.698413,0.984288,0.389429,0.536549,0.474418
ME_blanchedalmond,0.608428,0.013378,0.608428,0.234302,0.191290,0.634500,0.153517,0.354284,0.539560,0.338871
ME_brown,0.156178,0.246049,0.156178,0.608307,0.017658,0.980378,0.911020,0.426501,0.716601,0.495422
ME_burlywood,0.580918,0.901883,0.580918,0.417192,0.491953,0.417779,0.194777,0.028202,0.847071,0.743996
ME_chocolate,0.992602,0.129351,0.992602,0.395017,0.237605,0.204957,0.545967,0.583619,0.566090,0.892885
ME_coral,0.317682,0.160256,0.317682,0.754419,0.023238,0.430722,0.701752,0.407349,0.730880,0.082629
ME_darkgrey,0.002194,0.345553,0.002194,0.826866,0.527420,0.214743,0.157507,0.158220,0.000250,0.131131
ME_darkorange,0.793517,0.836217,0.793517,0.533489,0.646835,0.357638,0.375785,0.795351,0.960390,0.857642
ME_darkred,0.007839,0.217464,0.007839,0.646857,0.018255,0.000887,0.828147,0.390967,0.134812,0.051663


In [15]:
# Correção para múltiplos testes: Benjamini-Hochberg (FDR) sobre os 140 testes
# simultâneos (14 módulos × 10 traits). Sem correção, ~7 associações falsas seriam
# esperadas ao nível α=0.05 apenas por acaso.

padj_matrix_treated = pval_matrix_treated.copy().astype(float)

for trait_col in pval_matrix_treated.columns:
    pvals = pval_matrix_treated[trait_col].values.astype(float)
    valid_mask = ~np.isnan(pvals)
    if valid_mask.sum() > 0:
        _, padj, _, _ = multipletests(pvals[valid_mask], method="fdr_bh")
        padj_col = np.full(len(pvals), np.nan)
        padj_col[valid_mask] = padj
        padj_matrix_treated[trait_col] = padj_col

print("Matriz de p-valores ajustados (BH FDR):", padj_matrix_treated.shape)
display(padj_matrix_treated)

Matriz de p-valores ajustados (BH FDR): (32, 10)


,particle_size_um,concentration_gL,is_100nm,group_MA1,group_MA100,group_MB1,group_MB100,group_MC1,group_MC100,group_MD1
ME_antiquewhite,0.884426,0.948821,0.884426,0.876129,0.792861,0.801924,0.984288,0.884174,0.977679,0.755558
ME_bisque,0.884581,0.948821,0.884581,0.876129,0.590243,0.859585,0.984288,0.735994,0.977679,0.787119
ME_blanchedalmond,0.884426,0.428099,0.884426,0.876129,0.441344,0.836979,0.984288,0.735994,0.977679,0.787119
ME_brown,0.713958,0.745261,0.713958,0.876129,0.100581,0.980378,0.984288,0.735994,0.977679,0.787119
ME_burlywood,0.884426,0.950107,0.884426,0.876129,0.767157,0.810771,0.984288,0.256189,0.977679,0.874892
ME_chocolate,0.992602,0.745261,0.992602,0.876129,0.505390,0.635749,0.984288,0.753550,0.977679,0.952411
ME_coral,0.864116,0.745261,0.864116,0.876129,0.100581,0.810771,0.984288,0.735994,0.977679,0.661034
ME_darkgrey,0.061993,0.753217,0.061993,0.912404,0.767157,0.635749,0.984288,0.670023,0.008005,0.699367
ME_darkorange,0.884581,0.948821,0.884581,0.876129,0.792861,0.810771,0.984288,0.877629,0.987859,0.946364
ME_darkred,0.062709,0.745261,0.062709,0.876129,0.100581,0.023995,0.984288,0.735994,0.977679,0.551074


In [16]:
fig = px.imshow(
    corr_matrix_treated.astype(float),
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    text_auto='.2f',
    aspect="auto",
    labels=dict(x="Traits", y="Módulos", color="Correlação"),
    x=corr_matrix_treated.columns,
    y=corr_matrix_treated.index,
    title="Correlação entre module eigengenes e traits (somente tratadas)"
)

fig.update_layout(
    width=900, height=max(400, 35*len(corr_matrix_treated)),
    margin=dict(l=100, r=20, t=60, b=100),
    xaxis_title="Traits",
    yaxis_title="Módulos"
)

fig.show()

# Salva o heatmap como PNG (requer kaleido: pip install kaleido)
fig.write_image(str(HEATMAP_PATH))
print("Heatmap salvo em:", HEATMAP_PATH)

Heatmap salvo em: ../../data/interim/wgcna/wgcna_module_trait_heatmap_treated.png


## Consolidação dos Resultados do WGCNA

In [17]:
# Salvamento dos resultados sem controles

module_eigengenes_treated.to_csv(MODULE_EIGENGENES_TREATED_PATH)
corr_matrix_treated.to_csv(MODULE_TRAIT_CORR_PATH)
pval_matrix_treated.to_csv(MODULE_TRAIT_PVAL_PATH)
padj_matrix_treated.to_csv(MODULE_TRAIT_PADJ_PATH)

print("Arquivos salvos:")
print("-", MODULE_EIGENGENES_TREATED_PATH)
print("-", MODULE_TRAIT_CORR_PATH)
print("-", MODULE_TRAIT_PVAL_PATH)
print("-", MODULE_TRAIT_PADJ_PATH)

Arquivos salvos:
- ../../data/interim/wgcna/wgcna_module_eigengenes_treated.csv
- ../../data/interim/wgcna/wgcna_module_trait_correlations_treated.csv
- ../../data/interim/wgcna/wgcna_module_trait_pvalues_treated.csv
- ../../data/interim/wgcna/wgcna_module_trait_padj_treated.csv


In [18]:
# Resumo dos módulos mais associados aos traços de interesse.
# Inclui correlação, p-valor bruto e p-valor ajustado por BH (FDR).
# O flag `significant_*` usa padj < 0.05.

summary_rows = []

for module_name in corr_matrix_treated.index:
    row = {"module": module_name}

    for trait in ["particle_size_um", "concentration_gL", "is_100nm"]:
        if trait in corr_matrix_treated.columns:
            row[f"corr_{trait}"]        = corr_matrix_treated.loc[module_name, trait]
            row[f"pval_{trait}"]        = pval_matrix_treated.loc[module_name, trait]
            row[f"padj_{trait}"]        = padj_matrix_treated.loc[module_name, trait]
            row[f"significant_{trait}_padj005"] = padj_matrix_treated.loc[module_name, trait] < 0.05
            row[f"significant_{trait}_padj010"] = padj_matrix_treated.loc[module_name, trait] < 0.10

    summary_rows.append(row)

module_summary_df = pd.DataFrame(summary_rows)
module_summary_df.to_csv(MODULE_SUMMARY_PATH, index=False)

print("Resumo dos módulos salvo em:", MODULE_SUMMARY_PATH)
display(module_summary_df.sort_values("pval_particle_size_um", na_position="last").head(15))

Resumo dos módulos salvo em: ../../data/interim/wgcna/wgcna_module_summary.csv


,module,corr_particle_size_um,pval_particle_size_um,padj_particle_size_um,significant_particle_size_um_padj005,significant_particle_size_um_padj010,corr_concentration_gL,pval_concentration_gL,padj_concentration_gL,significant_concentration_gL_padj005,significant_concentration_gL_padj010,corr_is_100nm,pval_is_100nm,padj_is_100nm,significant_is_100nm_padj005,significant_is_100nm_padj010
7,ME_darkgrey,0.630282,0.002194,0.061993,False,True,0.216641,0.345553,0.753217,False,False,-0.630282,0.002194,0.061993,False,True
30,ME_tan,0.594107,0.004513,0.061993,False,True,-0.095091,0.681800,0.948591,False,False,-0.594107,0.004513,0.061993,False,True
15,ME_linen,-0.580357,0.005812,0.061993,False,True,0.340841,0.130537,0.745261,False,False,0.580357,0.005812,0.061993,False,True
9,ME_darkred,-0.563291,0.007839,0.062709,False,True,0.280868,0.217464,0.745261,False,False,0.563291,0.007839,0.062709,False,True
24,ME_red,-0.344660,0.126007,0.687598,False,False,0.306225,0.176993,0.745261,False,False,0.344660,0.126007,0.687598,False,False
29,ME_silver,-0.342190,0.128925,0.687598,False,False,0.043432,0.851715,0.948821,False,False,0.342190,0.128925,0.687598,False,False
3,ME_brown,0.320837,0.156178,0.713958,False,False,-0.264797,0.246049,0.745261,False,False,-0.320837,0.156178,0.713958,False,False
31,ME_white,0.266194,0.243471,0.864116,False,False,0.128758,0.578049,0.924878,False,False,-0.266194,0.243471,0.864116,False,False
13,ME_gainsboro,-0.263653,0.248170,0.864116,False,False,0.426001,0.054162,0.745261,False,False,0.263653,0.248170,0.864116,False,False
6,ME_coral,0.229163,0.317682,0.864116,False,False,-0.317872,0.160256,0.745261,False,False,-0.229163,0.317682,0.864116,False,False


## Nota Metodológica — Threshold de Significância (α)

**Para discussão com o grupo:**

Os testes de associação módulo-trait foram corrigidos para múltiplos testes pelo método Benjamini-Hochberg (FDR). Com n=21 amostras tratadas e o número de módulos gerado, nenhum módulo atingiu o threshold convencional de **padj < 0.05**.

Contudo, em estudos exploratórios de genômica com amostras pequenas, **α = 0.10 (FDR 10%) é aceito na literatura** como critério para seleção de módulos candidatos. Isso significa: entre todos os resultados chamados significativos, esperamos no máximo 10% de falsos positivos.

**Decisão a tomar antes de prosseguir para enriquecimento funcional:**
- Usar **α = 0.05** → nenhum módulo FDR-significativo; análise downstream baseada em significância nominal (p < 0.05 bruto)
- Usar **α = 0.10** → módulos com `padj < 0.10` são considerados significativos; justificativa metodológica deve ser explicitada na seção de métodos

Os flags `significant_*_padj005` e `significant_*_padj010` estão ambos disponíveis em `wgcna_module_summary.csv` para facilitar essa decisão.

## Comparação de Parâmetros

In [19]:
# Comparação entre a configuração original (minModuleSize=50, MEDissThres=0.2)
# arquivada em v_minmod50/ e a configuração atual definida nas constantes deste notebook.
# Execute esta célula após rodar o notebook completo com os novos parâmetros.

import pandas as pd

BASELINE_DIR = WGCNA_DIR / "v_minmod50"
CURRENT_LABEL = f"minmod{MIN_MODULE_SIZE} / meds{int(ME_DISS_THRES * 100)} (atual)"
BASELINE_LABEL = "minmod50 / meds20 (original)"


def summarize_run(label, run_dir):
    summary_path = run_dir / "wgcna_module_summary.csv"
    modules_path = run_dir / "wgcna_gene_modules.csv"

    if not summary_path.exists() or not modules_path.exists():
        return {"versão": label, "dimgrey (n / %)": "—", "nº módulos": "—",
                "melhor p-val particle_size": "—", "melhor padj particle_size": "—",
                "módulos padj < 0.05": "—"}

    summary = pd.read_csv(summary_path)
    modules = pd.read_csv(modules_path)

    n_total = len(modules)
    n_dimgrey = (modules["module"] == "dimgrey").sum()
    pct = 100 * n_dimgrey / n_total
    n_modules = modules["module"].nunique()

    best_pval = summary["pval_particle_size_um"].min()
    best_padj = summary["padj_particle_size_um"].min()
    n_sig_005 = (summary["padj_particle_size_um"] < 0.05).sum()
    n_sig_010 = (summary["padj_particle_size_um"] < 0.10).sum()

    return {
        "versão": label,
        "dimgrey (n / %)": f"{n_dimgrey} / {pct:.1f}%",
        "nº módulos": n_modules,
        "melhor p-val particle_size": f"{best_pval:.4f}",
        "melhor padj particle_size": f"{best_padj:.4f}",
        "módulos padj < 0.05": n_sig_005,
        "módulos padj < 0.10": n_sig_010,
    }


rows = [
    summarize_run(BASELINE_LABEL, BASELINE_DIR),
    summarize_run(CURRENT_LABEL, WGCNA_DIR),
]

comparison_df = pd.DataFrame(rows).set_index("versão")
print("=== Comparação de parâmetros WGCNA ===")
display(comparison_df.T)

def list_sig_modules(label, run_dir, alpha=0.10):
    summary_path = run_dir / "wgcna_module_summary.csv"
    if not summary_path.exists():
        print(f"\n[{label}] wgcna_module_summary.csv não encontrado.")
        return
    summary = pd.read_csv(summary_path)
    if "padj_particle_size_um" not in summary.columns:
        print(f"\n[{label}] coluna padj_particle_size_um ausente.")
        return
    sig = (
        summary[summary["padj_particle_size_um"] < alpha]
        [["module", "corr_particle_size_um", "pval_particle_size_um", "padj_particle_size_um"]]
        .rename(columns={
            "module": "módulo",
            "corr_particle_size_um": "r",
            "pval_particle_size_um": "p-val",
            "padj_particle_size_um": "padj",
        })
        .sort_values("p-val")
        .reset_index(drop=True)
    )
    sig["r"]      = sig["r"].round(3)
    sig["p-val"]  = sig["p-val"].round(4)
    sig["padj"]   = sig["padj"].round(4)
    sig["direção"] = sig["r"].apply(lambda r: "↑ em 1µm" if r > 0 else "↑ em 100nm")
    print(f"\n=== Módulos com padj < {alpha} — {label} ===")
    display(sig)

list_sig_modules(BASELINE_LABEL, BASELINE_DIR)
list_sig_modules(CURRENT_LABEL, WGCNA_DIR)

=== Comparação de parâmetros WGCNA ===


versão,minmod50 / meds20 (original),minmod20 / meds25 (atual)
dimgrey (n / %),—,3847 / 31.6%
nº módulos,—,32
melhor p-val particle_size,—,0.0022
melhor padj particle_size,—,0.0620
módulos padj < 0.05,—,0
módulos padj < 0.10,NaN,4.0



[minmod50 / meds20 (original)] wgcna_module_summary.csv não encontrado.

=== Módulos com padj < 0.1 — minmod20 / meds25 (atual) ===


,módulo,r,p-val,padj,direção
0,ME_darkgrey,0.630,0.0022,0.0620,↑ em 1µm
1,ME_tan,0.594,0.0045,0.0620,↑ em 1µm
2,ME_linen,-0.580,0.0058,0.0620,↑ em 100nm
3,ME_darkred,-0.563,0.0078,0.0627,↑ em 100nm
